# Fase 2 — Comprensión de los datos (EDA) · Bananas

**Proyecto:** clasificación de calidad y tamaño de bananas · Algoritmos y Programación III (2026-2).
**Rúbrica que alimenta:** 6 (análisis e interpretación), 1 (sesgos del dataset) y 8 (documentación).

**Dataset:** *Fruit Quality Classification* (R. D. Park, Kaggle, CC0) — https://www.kaggle.com/datasets/ryandpark/fruit-quality-classification

**Preguntas que responde este notebook** (los lineamientos piden: distribución de clases, desbalance, calidad de imagen, variabilidad):
1. ¿Cuántas imágenes de banana hay por clase y cuántas son realmente bananas *distintas*? (clave para evitar fuga de datos)
2. ¿Se pueden agrupar las imágenes por banana física a partir de la marca de tiempo del nombre de archivo? **(hipótesis, se valida visualmente)**
3. ¿Qué tan homogéneas son las imágenes (tamaño, color, nitidez)? ¿Hay atajos (p. ej. brillo o fondo) que delaten la clase?
4. ¿Hay duplicados, o imágenes idénticas con etiquetas distintas?
5. ¿Qué contiene `Mixed` para banana y cuánto nos sirve para construir la clase **Regular**?

> Las celdas no inventan resultados: todo lo que se afirme en el informe debe salir de la **ejecución local** de este notebook sobre el dataset real.

In [ ]:
import os, sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sys.path.insert(0, str(Path("..") / "src"))
import data_utils as du
import splits

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Ruta del dataset (se puede cambiar con la variable de entorno FRUIT_DATA_ROOT)
DATA_ROOT = Path(os.environ.get("FRUIT_DATA_ROOT", "../data/raw/Dataset Fruit"))
FOLDERS = {
    "kaggle_good":  DATA_ROOT / "Good Quality_Fruits" / "Banana_Good",
    "kaggle_bad":   DATA_ROOT / "Bad Quality_Fruits" / "Banana_Bad",
    "kaggle_mixed": DATA_ROOT / "Mixed Qualit_Fruits" / "Banana",   # 'Qualit' es el nombre original
}
LABELS = {"kaggle_good": "Buena", "kaggle_bad": "Mala", "kaggle_mixed": "Mixed"}
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("../data/processed"); OUT_DIR.mkdir(parents=True, exist_ok=True)

GAP_SECONDS = 3.0      # HIPÓTESIS: fotos de una misma banana distan <= 3 s (se valida abajo)
DUP_HASH_SIZE, DUP_MAX_DIST = 16, 2   # aHash de 256 bits; umbral ESTRICTO (fondo blanco => muchos falsos positivos con umbrales altos)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

## 1. Inventario

In [ ]:
frames = {k: du.list_images(v) for k, v in FOLDERS.items()}
inventory = pd.DataFrame({
    "carpeta": [str(v.relative_to(DATA_ROOT)) for v in FOLDERS.values()],
    "n_imagenes": [len(f) for f in frames.values()],
    "con_marca_de_tiempo": [int(f["timestamp"].notna().sum()) for f in frames.values()],
    "con_sufijo_NN": [int((f["dup_suffix"] > 0).sum()) for f in frames.values()],
}, index=list(FOLDERS))
display(inventory)
print("Fechas de captura por carpeta:")
for k, f in frames.items():
    d = f["timestamp"].dropna().dt.date.value_counts().sort_index()
    print(f"  {k:13s}", dict(d))

**Lectura esperada:** Buena y Mala deberían estar casi balanceadas entre sí. La clase **Regular no existe** en el dataset
(`Mixed` no es una calidad intermedia) y habrá que construirla. Si Good y Bad se capturaron en **fechas distintas**,
existe un posible sesgo de sesión (iluminación, fondo): se revisa en la sección 4.

## 2. Marcas de tiempo y agrupación por banana (hipótesis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, key in zip(axes, ["kaggle_good", "kaggle_bad"]):
    gaps = du.consecutive_gaps(frames[key])
    ax.hist(np.clip(gaps, 0, 60), bins=np.arange(0, 61, 1), color="#5454E9")
    ax.set_yscale("log")
    ax.set_xlabel("Tiempo entre fotos consecutivas (s, recortado a 60)")
    ax.set_ylabel("Número de pares de fotos")
    ax.set_title(f"{LABELS[key]} ({key})")
fig.tight_layout(); fig.savefig(FIG_DIR / "gaps_entre_fotos.pdf"); plt.show()

Si la hipótesis es correcta, se ve un pico a 0–2 s (ángulos de la misma fruta) y una cola de saltos largos (cambio de fruta).
La elección de `GAP_SECONDS` se apoya en esa separación y en la validación visual de más abajo.

In [ ]:
indexed = {}
for key, prefix in [("kaggle_good", "good_"), ("kaggle_bad", "bad_"), ("kaggle_mixed", "mixed_")]:
    indexed[key] = du.assign_groups(frames[key], gap_seconds=GAP_SECONDS, prefix=prefix)

summary = pd.DataFrame({
    k: {"imagenes": len(v), "grupos_estimados": v["group"].nunique(),
        "imagenes_por_grupo": round(len(v) / v["group"].nunique(), 2),
        "max_grupo": int(v["group_size"].max())}
    for k, v in indexed.items()}).T
display(summary)

print("Sensibilidad del número de grupos al umbral (s):")
for key in ("kaggle_good", "kaggle_bad"):
    print(key); display(du.group_sensitivity(frames[key]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, key in zip(axes, ["kaggle_good", "kaggle_bad"]):
    sizes = indexed[key].groupby("group").size().value_counts().sort_index()
    ax.bar(sizes.index.astype(str), sizes.values, color="#4CB87B")
    ax.set_xlabel("Imágenes por grupo"); ax.set_ylabel("Número de grupos"); ax.set_title(LABELS[key])
fig.tight_layout(); fig.savefig(FIG_DIR / "tamano_de_grupos.pdf"); plt.show()

### 2.1 Validación visual de los grupos (obligatoria)
Cada fila debe mostrar **la misma banana** desde distintos ángulos. Si aparecen bananas distintas en una fila (umbral muy grande)
o una misma banana partida en varias filas (umbral muy pequeño), hay que ajustar `GAP_SECONDS` y repetir.

In [ ]:
def show_groups(key, n=4, seed=SEED, max_imgs=8):
    df = indexed[key]
    candidates = df.loc[df["group_size"] >= 3, "group"].unique()
    if len(candidates) == 0:
        print("No hay grupos de 3+ imágenes: la hipótesis temporal no se sostiene para", key); return
    chosen = np.random.default_rng(seed).choice(candidates, size=min(n, len(candidates)), replace=False)
    fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 1.9 * len(chosen)))
    for ax, g in zip(np.atleast_1d(axes), chosen):
        paths = df.loc[df["group"] == g, "path"].tolist()[:max_imgs]
        ax.imshow(du.contact_sheet(paths, ncols=len(paths), thumb=140)); ax.axis("off")
        ax.set_title(f"{g}  ({int((df['group'] == g).sum())} imágenes)", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"grupos_{key}.png", dpi=150); plt.show()

show_groups("kaggle_good"); show_groups("kaggle_bad")

## 3. Calidad y variabilidad de las imágenes

In [ ]:
rows = []
for key, df in indexed.items():
    s = pd.DataFrame([du.image_stats(p) for p in df["path"]])
    s["source"], s["group"], s["path"] = key, df["group"].values, df["path"].values
    rows.append(s)
stats = pd.concat(rows, ignore_index=True)

print("Tamaños de imagen (ancho x alto) por fuente:")
display(stats.groupby(["source", "width", "height"]).size().rename("n").reset_index())
display(stats.groupby("source")[["r", "g", "b", "hue", "sat", "val", "sharpness"]].mean().round(3))

In [ ]:
cols = [("hue", "Tono medio (grados)"), ("sat", "Saturación media (0 a 1)"),
        ("val", "Brillo medio (0 a 1)"), ("sharpness", "Nitidez (varianza de gradientes)")]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (col, lab) in zip(axes, cols):
    ax.boxplot([stats.loc[stats.source == s, col] for s in ("kaggle_good", "kaggle_bad")])
    ax.set_xticks([1, 2]); ax.set_xticklabels(["Buena", "Mala"]); ax.set_ylabel(lab)
fig.tight_layout(); fig.savefig(FIG_DIR / "estadisticos_color.pdf"); plt.show()

## 4. ¿Atajos o sesgos? Línea base trivial con solo color global
Se entrena una regresión logística con **seis números por imagen** (R, G, B, tono, saturación, brillo medios), usando la
**partición por grupo**. Si esto ya separa Buena de Mala casi perfectamente, el contraste Buena/Mala del dataset es "fácil"
(o hay un atajo de sesión/fondo) y el reto real es **Regular** y el **cambio de dominio** hacia las fotos propias.

In [ ]:
bin_df = stats[stats.source.isin(["kaggle_good", "kaggle_bad"])].copy()
bin_df["label"] = bin_df["source"].map(LABELS)
train, test = splits.grouped_split(bin_df, label_col="label", group_col="group", test_size=0.2, seed=SEED)
feats = ["r", "g", "b", "hue", "sat", "val"]
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(train[feats], train["label"])
pred = clf.predict(test[feats])
print(f"Imágenes: train={len(train)}  test={len(test)} | grupos train={train.group.nunique()}  test={test.group.nunique()}")
print(f"Accuracy={accuracy_score(test['label'], pred):.3f}  F1-macro={f1_score(test['label'], pred, average='macro'):.3f}")
print("Error estándar binomial de la accuracy (n test imágenes, aproximado):",
      round(float(np.sqrt(accuracy_score(test['label'], pred) * (1 - accuracy_score(test['label'], pred)) / len(test))), 3),
      "— ojo: las imágenes de un mismo grupo no son independientes, el error real es mayor.")

## 5. Duplicados y etiquetas contradictorias

In [ ]:
hashes, meta = {}, []
for key in ("kaggle_good", "kaggle_bad"):
    for _, r in indexed[key].iterrows():
        hashes[r["path"]] = du.ahash(r["path"], size=DUP_HASH_SIZE)
        meta.append({"path": r["path"], "source": key, "group": r["group"]})
meta = pd.DataFrame(meta).set_index("path")
pairs = du.near_duplicates(hashes, max_dist=DUP_MAX_DIST)
if not pairs:
    print("No se encontraron pares con distancia <=", DUP_MAX_DIST)
    dup = pd.DataFrame(columns=["a", "b", "dist", "src_a", "src_b", "same_group"])
else:
    dup = pd.DataFrame(pairs, columns=["a", "b", "dist"])
    dup["src_a"], dup["src_b"] = meta.loc[dup.a, "source"].values, meta.loc[dup.b, "source"].values
    dup["same_group"] = meta.loc[dup.a, "group"].values == meta.loc[dup.b, "group"].values
    display(dup.groupby(["src_a", "src_b", "same_group"]).size().rename("pares_candidatos").reset_index())
cross = dup[dup.src_a != dup.src_b].sort_values("dist")
print(f"Pares candidatos entre Buena y Mala (posible etiqueta contradictoria): {len(cross)}")

In [ ]:
if len(cross):
    top = cross.head(4)
    fig, axes = plt.subplots(len(top), 1, figsize=(5, 2.2 * len(top)))
    for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
        ax.imshow(du.contact_sheet([r.a, r.b], ncols=2, thumb=180)); ax.axis("off")
        ax.set_title(f"{r.src_a} vs {r.src_b} · distancia hash = {r.dist}", fontsize=8)
    fig.tight_layout(); plt.show()

**Nota:** como todas las fotos son una fruta sobre fondo claro, el hash perceptual colisiona con facilidad: umbrales altos producen muchos falsos positivos. Por eso el umbral es estricto (`DUP_MAX_DIST`). Los pares dentro de un mismo grupo suelen ser copias `_NN` de la misma toma; los pares **entre Buena y Mala** son *candidatos* y se deben revisar a ojo antes de concluir que hay etiquetas contradictorias.

## 6. Carpeta `Mixed` (banana): ¿sirve para la clase Regular?

In [ ]:
mixed = indexed["kaggle_mixed"]
display(stats[stats.source == "kaggle_mixed"][["width", "height"]].value_counts().rename("n").reset_index())
sample = mixed.sample(n=min(12, len(mixed)), random_state=SEED)["path"].tolist()
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(du.contact_sheet(sample, ncols=4, thumb=240)); ax.axis("off")
ax.set_title("Muestra aleatoria de Mixed / Banana"); fig.tight_layout()
fig.savefig(FIG_DIR / "mixed_banana_muestra.png", dpi=150); plt.show()

**Para completar mirando la figura** (esto NO se puede decidir por código): ¿cuántas bananas hay por foto? ¿qué calidades aparecen?
¿hay que segmentarlas una por una (lineamientos §2.1)? Anotar aquí la conclusión y el plan de segmentación.

## 7. Índice de datos para las fases siguientes

In [ ]:
index = pd.concat([
    indexed[k].assign(source=k, label=LABELS[k])[["path", "filename", "timestamp", "source", "label", "group", "group_size"]]
    for k in indexed], ignore_index=True)
index.to_csv(OUT_DIR / "banana_index.csv", index=False)
print("Guardado:", (OUT_DIR / "banana_index.csv").resolve(), "| filas:", len(index))
print(index.groupby(["source", "label"]).agg(imagenes=("path", "size"), grupos=("group", "nunique")))

## 8. Hallazgos y decisiones — **completar tras ejecutar con los datos reales**

| Pregunta | Resultado observado | Decisión |
|---|---|---|
| Imágenes por clase y desbalance Buena/Mala | _(completar)_ | _(completar)_ |
| ¿La agrupación por tiempo identifica bananas? (validación visual) | _(completar)_ | umbral `GAP_SECONDS` = _(completar)_ |
| Nº de bananas distintas por clase | _(completar)_ | tamaño de test por grupos |
| Sesgo de sesión (fechas, brillo, fondo) | _(completar)_ | _(completar)_ |
| Línea base con color global (F1 macro) | _(completar)_ | _(completar)_ |
| Duplicados / etiquetas contradictorias | _(completar)_ | _(completar)_ |
| Utilidad de `Mixed` para la clase Regular | _(completar)_ | _(completar)_ |

Estos resultados alimentan `docs/formulacion_matematica.md` (§7, preguntas abiertas 1 y 2) y el informe (sección de datos y sesgos éticos).